In [ ]:
import os, re, csv, time, math, random, warnings, itertools
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional
from contextlib import nullcontext

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score

from PIL import Image
from tqdm.auto import tqdm


DRIVE_DATA_DIR = "/content/drive/MyDrive/brain_tumor_data_set"


USE_LOCAL_COPY = True
LOCAL_DATA_DIR = "/content/brain_tumor_data_set"

if USE_LOCAL_COPY:
    if not os.path.exists(LOCAL_DATA_DIR):
        os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
        !rsync -a "{DRIVE_DATA_DIR}/" "{LOCAL_DATA_DIR}/"
    DATA_DIR = LOCAL_DATA_DIR
else:
    DATA_DIR = DRIVE_DATA_DIR

assert os.path.isdir(DATA_DIR), f"DATA_DIR not found: {DATA_DIR}"
print("Using DATA_DIR =", DATA_DIR)


OUT_DIR = "/content/drive/MyDrive/chapter5_planar_factorial"
os.makedirs(OUT_DIR, exist_ok=True)
RESULTS_CSV = os.path.join(OUT_DIR, "results_factorial_cv.csv")
ANALYSIS_DIR = os.path.join(OUT_DIR, "analysis_out")
os.makedirs(ANALYSIS_DIR, exist_ok=True)

print("Results CSV:", RESULTS_CSV)
print("Analysis dir:", ANALYSIS_DIR)


IMG_SIZE = 224
BATCH_SIZE = 64
MAX_EPOCHS = 20

BASE_LR = 1e-3
WEIGHT_DECAY = 0.0
DROPOUT = 0.0


K_FOLDS = 5
SEEDS = [0, 1, 2]   


USE_EARLY_STOP = True
PATIENCE = 3
MIN_DELTA = 1e-4


NUM_WORKERS = 2
PIN_MEMORY = True


USE_AMP = True


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE =", DEVICE)


warnings.filterwarnings("ignore", category=FutureWarning)


torch.backends.cudnn.benchmark = True


Using DATA_DIR = /content/brain_tumor_data_set
Results CSV: /content/drive/MyDrive/chapter5_planar_factorial/results_factorial_cv.csv
Analysis dir: /content/drive/MyDrive/chapter5_planar_factorial/analysis_out
DEVICE = cuda


In [ ]:
from torchvision import transforms  

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id: int):
    
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def build_transforms(img_size: int = 224):
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5)),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5)),
    ])
    return train_tf, val_tf

def infer_pos_class_idx(class_to_idx: Dict[str,int]) -> int:
    # prefer common names
    lower = {k.lower(): v for k, v in class_to_idx.items()}
    for key in ["yes", "tumor", "positive", "1"]:
        if key in lower:
            return int(lower[key])
    # fallback: take max index as positive
    return int(max(lower.values()))

class BinaryImagePathDataset(Dataset):
    """
    Dataset based on a fixed sample list (path, class_idx), mapping to binary label:
      y = 1 if class_idx == pos_class_idx else 0
    """
    def __init__(self, samples: List[Tuple[str,int]], pos_class_idx: int, transform=None, cache_images: bool=True):
        self.samples = samples
        self.pos_class_idx = int(pos_class_idx)
        self.transform = transform

        self.cache_images = bool(cache_images)
        self._cache = None
        if self.cache_images:
            # preload to RAM (dataset is tiny: 253 images)
            cache = []
            for p, _ in tqdm(self.samples, desc="Caching images to RAM", leave=False):
                with Image.open(p) as img:
                    cache.append(img.convert("RGB").copy())
            self._cache = cache

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, cls = self.samples[idx]
        y = 1 if int(cls) == self.pos_class_idx else 0

        if self._cache is not None:
            img = self._cache[idx].copy()
        else:
            with Image.open(path) as im:
                img = im.convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        return img, y

# ---- scan dataset once ----
from torchvision import datasets
tmp_ds = datasets.ImageFolder(DATA_DIR)  # only to get samples + class_to_idx
samples = tmp_ds.samples
class_to_idx = tmp_ds.class_to_idx
pos_class_idx = infer_pos_class_idx(class_to_idx)

print("class_to_idx =", class_to_idx)
print("pos_class_idx =", pos_class_idx)

targets_bin = np.array([1 if cls == pos_class_idx else 0 for _, cls in samples], dtype=np.int64)
print("N samples =", len(samples), "Pos =", targets_bin.sum(), "Neg =", (targets_bin==0).sum())

train_tf, val_tf = build_transforms(IMG_SIZE)

# Use caching to speed up repeated CV
CACHE_IMAGES_IN_RAM = True
full_train_ds = BinaryImagePathDataset(samples, pos_class_idx, transform=train_tf, cache_images=CACHE_IMAGES_IN_RAM)
full_val_ds   = BinaryImagePathDataset(samples, pos_class_idx, transform=val_tf,   cache_images=CACHE_IMAGES_IN_RAM)


class_to_idx = {'no': 0, 'yes': 1}
pos_class_idx = 1
N samples = 253 Pos = 155 Neg = 98


Caching images to RAM:   0%|          | 0/253 [00:00<?, ?it/s]

Caching images to RAM:   0%|          | 0/253 [00:00<?, ?it/s]

In [ ]:
def get_activation(name: str) -> nn.Module:
    name = name.lower()
    if name == "relu":
        return nn.ReLU(inplace=True)
    if name == "leakyrelu":
        return nn.LeakyReLU(negative_slope=0.1, inplace=True)
    if name == "sigmoid":
        return nn.Sigmoid()
    raise ValueError(f"Unknown activation: {name}")

class PlanarCNN(nn.Module):
    """
    Conv blocks: (Conv2d -> Activation -> MaxPool2d)
    channels: 3 -> 32 -> 64 -> 128 (depending on conv_layers)
    FC: hidden dim fixed 64, output 1 logit
    """
    def __init__(self, conv_layers: int, fc_layers: int, activation: str,
                 input_size: int = 224, in_channels: int = 3,
                 base_channels: int = 32, hidden_dim: int = 64,
                 dropout: float = 0.0):
        super().__init__()
        assert conv_layers in [1,2,3]
        assert fc_layers in [1,2,3]

        conv = []
        c_in = in_channels
        c_out = base_channels
        for _ in range(conv_layers):
            conv.append(nn.Conv2d(c_in, c_out, kernel_size=3, padding=1))
            conv.append(get_activation(activation))
            conv.append(nn.MaxPool2d(kernel_size=2, stride=2))
            c_in = c_out
            c_out *= 2
        self.conv = nn.Sequential(*conv)

        # infer flatten dim
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, input_size, input_size)
            feat = self.conv(dummy)
            flat_dim = int(np.prod(feat.shape[1:]))

        fc = []
        if fc_layers == 1:
            fc.append(nn.Linear(flat_dim, 1))
        else:
            fc.append(nn.Linear(flat_dim, hidden_dim))
            fc.append(get_activation(activation))
            if dropout > 0:
                fc.append(nn.Dropout(dropout))

            if fc_layers == 3:
                fc.append(nn.Linear(hidden_dim, hidden_dim))
                fc.append(get_activation(activation))
                if dropout > 0:
                    fc.append(nn.Dropout(dropout))

            fc.append(nn.Linear(hidden_dim, 1))

        self.fc = nn.Sequential(*fc)

    def forward(self, x):
        x = self.conv(x)
        x = torch.flatten(x, 1)
        x = self.fc(x).squeeze(1)  # (B,)
        return x

def build_optimizer(name: str, params, lr: float, weight_decay: float):
    name = name.lower()
    if name == "adam":
        return optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "sgd":
        return optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    if name == "rmsprop":
        return optim.RMSprop(params, lr=lr, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {name}")

def safe_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    try:
        return float(roc_auc_score(y_true, y_prob))
    except ValueError:
        return float("nan")

def compute_metrics(y_true: np.ndarray, logits: np.ndarray) -> Dict[str,float]:
    probs = 1.0 / (1.0 + np.exp(-logits))
    preds = (probs >= 0.5).astype(np.int64)
    return {
        "auc": safe_auc(y_true, probs),
        "acc": float(accuracy_score(y_true, preds)),
        "bal_acc": float(balanced_accuracy_score(y_true, preds)),
        "f1": float(f1_score(y_true, preds, zero_division=0)),
    }

# ---- AMP wrappers (new API first; fallback if needed) ----
def get_amp_dtype():
    if DEVICE.type != "cuda":
        return None
    try:
        if torch.cuda.is_bf16_supported():
            return torch.bfloat16
    except Exception:
        pass
    return torch.float16

AMP_DTYPE = get_amp_dtype()

def autocast_ctx():
    if not USE_AMP or DEVICE.type != "cuda":
        return nullcontext()
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast("cuda", dtype=AMP_DTYPE)
    return torch.cuda.amp.autocast(dtype=AMP_DTYPE)

def make_grad_scaler():
    use_scaler = (USE_AMP and DEVICE.type=="cuda" and AMP_DTYPE==torch.float16)
    if not use_scaler:
        return None
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        try:
            return torch.amp.GradScaler("cuda")
        except TypeError:
            return torch.amp.GradScaler()
    return torch.cuda.amp.GradScaler()

def train_one_epoch(model, loader, optimizer, criterion, scaler=None):
    model.train()
    losses = []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True).float()

        optimizer.zero_grad(set_to_none=True)

        with autocast_ctx():
            logits = model(x)
            loss = criterion(logits, y)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        losses.append(loss.item())
    return float(np.mean(losses)) if losses else float("nan")

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    all_logits, all_y = [], []
    losses = []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True).float()

        logits = model(x)
        loss = criterion(logits, y)
        losses.append(loss.item())

        all_logits.append(logits.detach().cpu().numpy())
        all_y.append(y.detach().cpu().numpy())

    logits_np = np.concatenate(all_logits).astype(np.float64)
    y_np = np.concatenate(all_y).astype(np.int64)
    metrics = compute_metrics(y_np, logits_np)
    return float(np.mean(losses)) if losses else float("nan"), metrics


In [ ]:
@dataclass(frozen=True)
class Config:
    optimizer: str
    activation: str
    conv_layers: int
    fc_layers: int

def enumerate_configs() -> List[Config]:
    OPTIMIZERS = ["adam", "sgd", "rmsprop"]
    ACTIVATIONS = ["relu", "leakyrelu", "sigmoid"]
    CONV_LAYERS = [1,2,3]
    FC_LAYERS = [1,2,3]
    return [Config(o,a,cl,fc) for o,a,cl,fc in itertools.product(OPTIMIZERS, ACTIVATIONS, CONV_LAYERS, FC_LAYERS)]

RESULT_FIELDS = [
    "optimizer","activation","conv_layers","fc_layers",
    "seed","fold",
    "best_epoch",
    "best_val_loss",
    "best_val_auc","best_val_bal_acc","best_val_acc","best_val_f1",
    "train_seconds",
    "n_params"
]

def load_done_keys(csv_path: str):
    if not os.path.exists(csv_path):
        return set()
    df = pd.read_csv(csv_path)
    keys = set()
    for _, r in df.iterrows():
        keys.add((str(r["optimizer"]), str(r["activation"]), int(r["conv_layers"]), int(r["fc_layers"]), int(r["seed"]), int(r["fold"])))
    return keys

def append_row(csv_path: str, row: Dict):
    file_exists = os.path.exists(csv_path)
    with open(csv_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=RESULT_FIELDS)
        if not file_exists:
            w.writeheader()
        w.writerow(row)

def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def run_one_fold(cfg: Config, seed: int, fold: int, train_idx: np.ndarray, val_idx: np.ndarray) -> Dict:
    set_global_seed(seed*1000 + fold)

    train_subset = Subset(full_train_ds, train_idx.tolist())
    val_subset   = Subset(full_val_ds,   val_idx.tolist())

    y_train = targets_bin[train_idx]
    n_pos = int(y_train.sum())
    n_neg = int((y_train == 0).sum())
    pos_weight_value = (n_neg / max(n_pos, 1))
    pos_weight = torch.tensor([pos_weight_value], device=DEVICE, dtype=torch.float32)

    g = torch.Generator()
    g.manual_seed(seed*1000 + fold)

    dl_kwargs = dict(
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE.type=="cuda"),
        worker_init_fn=seed_worker if NUM_WORKERS>0 else None,
        generator=g
    )

    train_loader = DataLoader(
        train_subset, batch_size=BATCH_SIZE, shuffle=True,
        persistent_workers=(NUM_WORKERS>0), **dl_kwargs
    )
    val_loader = DataLoader(
        val_subset, batch_size=BATCH_SIZE, shuffle=False,
        persistent_workers=(NUM_WORKERS>0), **dl_kwargs
    )

    model = PlanarCNN(cfg.conv_layers, cfg.fc_layers, cfg.activation, input_size=IMG_SIZE, dropout=DROPOUT).to(DEVICE)
    n_params = count_params(model)

    optimizer = build_optimizer(cfg.optimizer, model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    scaler = make_grad_scaler()

    best = {
        "epoch": -1,
        "val_loss": float("inf"),
        "auc": -float("inf"),
        "bal_acc": -float("inf"),
        "acc": -float("inf"),
        "f1": -float("inf")
    }

    patience_left = PATIENCE
    best_loss_so_far = float("inf")

    t0 = time.time()
    for epoch in range(1, MAX_EPOCHS+1):
        _ = train_one_epoch(model, train_loader, optimizer, criterion, scaler=scaler)
        val_loss, metrics = evaluate(model, val_loader, criterion)

        auc = metrics["auc"]
        if (not np.isnan(auc) and auc > best["auc"]) or (np.isclose(auc, best["auc"]) and metrics["bal_acc"] > best["bal_acc"]):
            best.update({
                "epoch": epoch,
                "val_loss": val_loss,
                "auc": metrics["auc"],
                "bal_acc": metrics["bal_acc"],
                "acc": metrics["acc"],
                "f1": metrics["f1"]
            })

        if USE_EARLY_STOP:
            if val_loss < (best_loss_so_far - MIN_DELTA):
                best_loss_so_far = val_loss
                patience_left = PATIENCE
            else:
                patience_left -= 1
                if patience_left <= 0:
                    break

        scheduler.step()

    train_seconds = time.time() - t0

    del model, optimizer, scheduler, train_loader, val_loader
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "optimizer": cfg.optimizer,
        "activation": cfg.activation,
        "conv_layers": cfg.conv_layers,
        "fc_layers": cfg.fc_layers,
        "seed": seed,
        "fold": fold,
        "best_epoch": int(best["epoch"]),
        "best_val_loss": float(best["val_loss"]),
        "best_val_auc": float(best["auc"]),
        "best_val_bal_acc": float(best["bal_acc"]),
        "best_val_acc": float(best["acc"]),
        "best_val_f1": float(best["f1"]),
        "train_seconds": float(train_seconds),
        "n_params": int(n_params)
    }


configs = enumerate_configs()
done = load_done_keys(RESULTS_CSV)

all_tasks = []
for cfg in configs:
    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=seed)
        for fold, (tr, va) in enumerate(skf.split(np.zeros_like(targets_bin), targets_bin)):
            key = (cfg.optimizer, cfg.activation, cfg.conv_layers, cfg.fc_layers, seed, fold)
            all_tasks.append((cfg, seed, fold, tr, va, key))

total = len(all_tasks)
missing = [t for t in all_tasks if t[-1] not in done]

print(f"Total target rows (config x seed x fold): {total}")
print(f"Already existing rows: {len(done)}")
print(f"Running missing folds: {len(missing)}")

pbar = tqdm(missing, desc="Running folds", unit="fold")
for cfg, seed, fold, tr, va, key in pbar:
    row = run_one_fold(cfg, seed, fold, tr, va)
    append_row(RESULTS_CSV, row)
    pbar.set_postfix({
        "opt": cfg.optimizer, "act": cfg.activation,
        "CL": cfg.conv_layers, "FC": cfg.fc_layers,
        "seed": seed, "fold": fold,
        "auc": f"{row['best_val_auc']:.3f}"
    })

print("Done. Results saved to:", RESULTS_CSV)


Total target rows (config x seed x fold): 1215
Already existing rows: 1215
Running missing folds: 0


Running folds: 0fold [00:00, ?fold/s]

Done. Results saved to: /content/drive/MyDrive/chapter5_planar_factorial/results_factorial_cv.csv


In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

df = pd.read_csv(RESULTS_CSV)

# basic cleaning
df["optimizer"] = df["optimizer"].astype(str).str.lower()
df["activation"] = df["activation"].astype(str).str.lower()
for c in ["conv_layers","fc_layers","seed","fold"]:
    df[c] = df[c].astype(int)

# choose metric for analysis 
METRIC = "best_val_auc"
df = df[np.isfinite(df[METRIC])].copy()

# ranking table
group_cols = ["optimizer","activation","conv_layers","fc_layers"]
rank_tbl = (df.groupby(group_cols)[METRIC]
              .agg(["mean","std","count"])
              .reset_index()
              .sort_values("mean", ascending=False))
rank_path = os.path.join(ANALYSIS_DIR, "config_ranking.csv")
rank_tbl.to_csv(rank_path, index=False)
print("Saved:", rank_path)
display(rank_tbl.head(10))

# main effects
for factor in ["optimizer","activation","conv_layers","fc_layers"]:
    me = (df.groupby([factor])[METRIC]
            .agg(["mean","std","count"])
            .reset_index()
            .sort_values("mean", ascending=False))
    me_path = os.path.join(ANALYSIS_DIR, f"main_effect_{factor}.csv")
    me.to_csv(me_path, index=False)
    print("Saved:", me_path)

# ANOVA with pairwise interactions + blocks (seed/fold as blocks)
formula = (
    f"{METRIC} ~ "
    "C(optimizer) + C(activation) + C(conv_layers) + C(fc_layers) + "
    "C(optimizer):C(activation) + C(optimizer):C(conv_layers) + C(optimizer):C(fc_layers) + "
    "C(activation):C(conv_layers) + C(activation):C(fc_layers) + "
    "C(conv_layers):C(fc_layers) + "
    "C(seed) + C(fold)"
)

model = smf.ols(formula, data=df).fit()
anova_tbl = sm.stats.anova_lm(model, typ=2)

anova_path = os.path.join(ANALYSIS_DIR, "anova_table.csv")
anova_tbl.to_csv(anova_path)
print("\nFormula:\n", formula)
print("\nANOVA (Type II):")
display(anova_tbl)
print("Saved:", anova_path)


Saved: /content/drive/MyDrive/chapter5_planar_factorial/analysis_out/config_ranking.csv


,optimizer,activation,conv_layers,fc_layers,mean,std,count
5,adam,leakyrelu,2,3,0.859174,0.069737,15
4,adam,leakyrelu,2,2,0.852920,0.066896,15
36,rmsprop,relu,1,1,0.852739,0.066132,15
0,adam,leakyrelu,1,1,0.851616,0.063992,15
37,rmsprop,relu,1,2,0.849194,0.064581,15
8,adam,leakyrelu,3,3,0.848936,0.072546,15
6,adam,leakyrelu,3,1,0.848925,0.068169,15
27,rmsprop,leakyrelu,1,1,0.848557,0.062299,15
29,rmsprop,leakyrelu,1,3,0.847586,0.069616,15
13,adam,relu,2,2,0.847051,0.069199,15


Saved: /content/drive/MyDrive/chapter5_planar_factorial/analysis_out/main_effect_optimizer.csv
Saved: /content/drive/MyDrive/chapter5_planar_factorial/analysis_out/main_effect_activation.csv
Saved: /content/drive/MyDrive/chapter5_planar_factorial/analysis_out/main_effect_conv_layers.csv
Saved: /content/drive/MyDrive/chapter5_planar_factorial/analysis_out/main_effect_fc_layers.csv

Formula:
 best_val_auc ~ C(optimizer) + C(activation) + C(conv_layers) + C(fc_layers) + C(optimizer):C(activation) + C(optimizer):C(conv_layers) + C(optimizer):C(fc_layers) + C(activation):C(conv_layers) + C(activation):C(fc_layers) + C(conv_layers):C(fc_layers) + C(seed) + C(fold)

ANOVA (Type II):


,sum_sq,df,F,PR(>F)
C(optimizer),0.241804,2.0,19.296820,5.676433e-09
C(activation),12.202665,2.0,973.814214,3.465752e-250
C(conv_layers),0.347970,2.0,27.769166,1.644826e-12
C(fc_layers),1.689898,2.0,134.859615,1.862678e-53
C(seed),0.013541,2.0,1.080598,3.397292e-01
C(fold),0.721705,4.0,28.797241,7.336678e-23
C(optimizer):C(activation),0.631811,4.0,25.210334,4.646430e-20
C(optimizer):C(conv_layers),0.117175,4.0,4.675465,9.543101e-04
C(optimizer):C(fc_layers),0.165668,4.0,6.610435,2.922830e-05
C(activation):C(conv_layers),0.266739,4.0,10.643351,1.786548e-08


Saved: /content/drive/MyDrive/chapter5_planar_factorial/analysis_out/anova_table.csv
